In [1]:
import os
import numpy as np
import cv2
import glob
from distutils.archive_util import make_archive
from imutils.object_detection import non_max_suppression
from multiprocessing import Pool
from tqdm.auto import tqdm

In [2]:
"""
src: https://pyimagesearch.com/2018/08/20/opencv-text-detection-east-text-detector/
"""
#source_dir = '../data/hateful_memes/img'
#source_dir = '../data/hateful_memes/img'
#target_dir_masked = '../data/hateful_memes_masked'
target_dir_inpainted = '../data/hateful_memes_inpainted'
source_dir = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img'
target_dir_masked = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/masked'
target_dir_inpainted = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/inpainted'
east_path = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/MODEL/frozen_east_text_detection.pb'

In [ ]:
model_width = 640  
model_height = 640
min_confidence = 0.4

In [4]:
print("[INFO] loading EAST text detector...")
net = cv2.dnn.readNet(east_path)
layerNames = [
    "feature_fusion/Conv_7/Sigmoid",
    "feature_fusion/concat_3"
]


[INFO] loading EAST text detector...


In [5]:
supported_formats = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']

image_files = []
for format in supported_formats:
    image_files.extend(glob.glob(os.path.join(source_dir, format)))

print(f"找到 {len(image_files)} 张图片需要处理")

找到 10000 张图片需要处理


In [6]:
def transform_image(img_fp):
    global model_width, model_height, min_confidence
    
    # load the input image and grab the image dimensions
    image = cv2.imread(img_fp)
    if image is None:
        print(f"无法读取图片: {img_fp}")
        return None
        
    masked = image.copy()
    (H, W) = image.shape[:2]
    
    # set the new width and height and then determine the ratio in change
    # for both the width and height
    (newW, newH) = (model_width, model_height)  # 修正变量名
    rW = W / float(newW)
    rH = H / float(newH)
    
    # resize the image and grab the new image dimensions
    image = cv2.resize(image, (newW, newH))
    (H, W) = image.shape[:2]

    # construct a blob from the image and then perform a forward pass of
    # the model to obtain the two output layer sets
    blob = cv2.dnn.blobFromImage(image, 1.0, (W, H),
        (123.68, 116.78, 103.94), swapRB=True, crop=False)
    net.setInput(blob)
    (scores, geometry) = net.forward(layerNames)

    # grab the number of rows and columns from the scores volume, then
    # initialize our set of bounding box rectangles and corresponding
    # confidence scores
    (numRows, numCols) = scores.shape[2:4]
    rects = []
    confidences = []
    
    # loop over the number of rows
    for y in range(0, numRows):
        # extract the scores (probabilities), followed by the geometrical
        # data used to derive potential bounding box coordinates that
        # surround text
        scoresData = scores[0, 0, y]
        xData0 = geometry[0, 0, y]
        xData1 = geometry[0, 1, y]
        xData2 = geometry[0, 2, y]
        xData3 = geometry[0, 3, y]
        anglesData = geometry[0, 4, y]

        # loop over the number of columns
        for x in range(0, numCols):
            # if our score does not have sufficient probability, ignore it
            if scoresData[x] < min_confidence:
                continue
            # compute the offset factor as our resulting feature maps will
            # be 4x smaller than the input image
            (offsetX, offsetY) = (x * 4.0, y * 4.0)
            # extract the rotation angle for the prediction and then
            # compute the sin and cosine
            angle = anglesData[x]
            cos = np.cos(angle)
            sin = np.sin(angle)
            # use the geometry volume to derive the width and height of
            # the bounding box
            h = xData0[x] + xData2[x]
            w = xData1[x] + xData3[x]
            # compute both the starting and ending (x, y)-coordinates for
            # the text prediction bounding box
            endX = int(offsetX + (cos * xData1[x]) + (sin * xData2[x]))
            endY = int(offsetY - (sin * xData1[x]) + (cos * xData2[x]))
            startX = int(endX - w)
            startY = int(endY - h)
            # add the bounding box coordinates and probability score to
            # our respective lists
            rects.append((startX, startY, endX, endY))
            confidences.append(scoresData[x])

    # 合并同一行的文本框
    if len(rects) > 0:
        rects = np.array(rects)
        
        # 计算每个框的中心y坐标和高度
        centers_y = (rects[:, 1] + rects[:, 3]) / 2
        heights = rects[:, 3] - rects[:, 1]
        avg_height = np.mean(heights)
        
        # 根据y坐标和高度将框分组到不同的行
        rows = []
        current_row = [rects[0]]
        for i in range(1, len(rects)):
            if abs(centers_y[i] - centers_y[i-1]) < avg_height / 2:
                current_row.append(rects[i])
            else:
                rows.append(current_row)
                current_row = [rects[i]]
        rows.append(current_row)
        
        merged_boxes = []
        for row in rows:
            row = np.array(row)
            min_x = np.min(row[:, 0])
            min_y = np.min(row[:, 1])
            max_x = np.max(row[:, 2])
            max_y = np.max(row[:, 3])
            merged_boxes.append([min_x, min_y, max_x, max_y])
        
        boxes = merged_boxes
    else:
        boxes = []

    # loop over the bounding boxes
    for (startX, startY, endX, endY) in boxes:
        # scale the bounding box coordinates based on the respective
        # ratios
        startX = int(startX * rW)
        startY = int(startY * rH)
        endX = int(endX * rW)
        endY = int(endY * rH)
        # draw the bounding box on the image
        cv2.rectangle(masked, (startX, startY), (endX, endY), (127, 127, 127), -1)

    return masked

In [7]:
from tqdm import tqdm

for i, img_path in enumerate(tqdm(image_files, desc="处理图片")):
    try:
        tqdm.write(f"处理第 {i+1}/{len(image_files)} 张图片: {os.path.basename(img_path)}")

        result_image = transform_image(img_path)
        
        if result_image is not None:
            filename = os.path.basename(img_path)
            name, ext = os.path.splitext(filename)
            output_path = os.path.join(target_dir_masked, f"{name}_masked{ext}")

            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            cv2.imwrite(output_path, result_image)
        
    except Exception as e:
        tqdm.write(f"处理图片 {img_path} 时出错: {str(e)}")
        continue

print("所有图片处理完成！")

处理图片:   0%|          | 11/10000 [00:00<01:43, 96.91it/s]    

处理第 1/10000 张图片: 01235.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01235.png 时出错: name 'model_width' is not defined
处理第 2/10000 张图片: 01236.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01236.png 时出错: name 'model_width' is not defined
处理第 3/10000 张图片: 01243.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01243.png 时出错: name 'model_width' is not defined
处理第 4/10000 张图片: 01245.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01245.png 时出错: name 'model_width' is not defined
处理第 5/10000 张图片: 01247.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01247.png 时出错: name 'model_width' is not defined
处理第 6/10000 张图片: 01256.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01256.png 时出错: name 'model_width' is not defined
处理第 7/10000 张图片: 01258.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Proje

处理图片:   0%|          | 11/10000 [00:00<01:43, 96.91it/s]    

处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01324.png 时出错: name 'model_width' is not defined
处理第 18/10000 张图片: 01325.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\01325.png 时出错: name 'model_width' is not defined
处理第 19/10000 张图片: 01327.png


处理图片:   0%|          | 30/10000 [00:00<01:54, 86.97it/s]    

...

处理图片:  19%|█▉        | 1946/10000 [00:28<01:38, 81.43it/s]    

处理第 1938/10000 张图片: 19364.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19364.png 时出错: name 'model_width' is not defined
处理第 1939/10000 张图片: 19367.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19367.png 时出错: name 'model_width' is not defined
处理第 1940/10000 张图片: 19370.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19370.png 时出错: name 'model_width' is not defined
处理第 1941/10000 张图片: 19372.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19372.png 时出错: name 'model_width' is not defined
处理第 1942/10000 张图片: 19375.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19375.png 时出错: name 'model_width' is not defined
处理第 1943/10000 张图片: 19380.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19380.png 时出错: name 'model_width' is not defined
处理第 1944/10000 张图片: 19384.png
处理图片 C:/Users/gidle/De

处理图片:  20%|█▉        | 1955/10000 [00:28<01:56, 69.07it/s]    

处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19408.png 时出错: name 'model_width' is not defined
处理第 1948/10000 张图片: 19420.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19420.png 时出错: name 'model_width' is not defined
处理第 1949/10000 张图片: 19426.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19426.png 时出错: name 'model_width' is not defined
处理第 1950/10000 张图片: 19427.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19427.png 时出错: name 'model_width' is not defined
处理第 1951/10000 张图片: 19430.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19430.png 时出错: name 'model_width' is not defined
处理第 1952/10000 张图片: 19438.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img\19438.png 时出错: name 'model_width' is not defined
处理第 1953/10000 张图片: 19452.png
处理图片 C:/Users/gidle/Desktop/Year3/NLP/Project/Hatefu

KeyboardInterrupt: 

def transform_and_save_image(img_fn):
    img_fp = os.path.join(source_dir, img_fn)
    img_fp_masked = os.path.join(target_dir_masked, img_fn)
    # img_fp_inpainted = os.path.join(target_dir_inpainted, img_fn)

    img_masked, img_inpainted = transform_image(img_fp)
    cv2.imwrite(img_fp_masked, img_masked)
    # cv2.imwrite(img_fp_inpainted, img_inpainted)

with Pool(64) as pool:
    #pool.map(transform_and_save_image, img_fns)
    for _ in tqdm(pool.imap_unordered(transform_and_save_image, img_fns), total=len(img_fns)):
        pass
